# Tujuan, Hipotesis, dan Data

1. Tujuan:
* Mencari apakah variabel X mempengaruhi variabel Y
* Mencari apakah variabel moderasi Xm mempengaruhi variabel Y
* Mencari apakah variabel X mempengaruhi variabel Y dimoderasi oleh Xm

2. Hipotesis:
* H0 = Tidak ada pengaruh antara X dengan Y
* H1 = Ada pengaruh antara X dengan Y
* H2 = Ada pengaruh antara Xm dengan Y
* H3 = Ada pengaruh antara X dengan variabel Y dimoderasi oleh Xm

3. Data:
* Data dibuat dengan dummy, tapi menggunakan seed. Jadi akan konsisten walaupun di-run beberapa kali. Masing-masing subbagian X, Xm, dan Y juga memiliki nilai yang didapat dari beberapa pertanyaan dan simulasi jawabannya dengan skala Likert.

Lebih disarankan untuk mempelajari terlebih dahulu `Simple-Regresi Linear.ipynb` dan `Advanced-Regresi Linear X dan Y.ipynb` karena ada panduan lengkap.

# Import dan Create Data

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import pearsonr, shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, linear_rainbow

In [2]:
# ==========================================
# 0. GENERATE DATA SURVEI IDEAL (100 Responden)
# ==========================================
np.random.seed(73)
n = 131

# Karakter dasar (Latent)
latent_x = np.random.normal(3.5, 0.6, n)
latent_xm = np.random.normal(3.2, 0.5, n) # Variabel Moderasi
# Y dipengaruhi X, Xm, dan Interaksi (X*Xm)
latent_y = (0.4 * latent_x) + (0.2 * latent_xm) + (0.15 * latent_x * latent_xm) + np.random.normal(0, 0.3, n)

def gen_items(latent, prefix, count=3):
    items = {}
    for i in range(1, count + 1):
        val = np.clip(np.round(latent + np.random.normal(0, 0.4, n)), 1, 5).astype(int)
        items[f"{prefix}_{i}"] = val
    return items

data = {}
data.update(gen_items(latent_x, 'X'))
data.update(gen_items(latent_xm, 'Xm'))
data.update(gen_items(latent_y, 'Y'))
df = pd.DataFrame(data)
df.sample(5)

,X_1,X_2,X_3,Xm_1,Xm_2,Xm_3,Y_1,Y_2,Y_3
48,4,3,4,3,4,3,4,4,5
25,4,4,5,3,2,3,4,3,4
8,3,2,3,2,3,3,3,3,3
2,5,5,5,3,4,3,5,5,5
43,2,2,3,3,3,3,3,3,3


# Uji Validitas dan Reliabilitas

In [3]:
def cronbach_alpha(df_items):
    k = df_items.shape[1]
    var_items = df_items.var(ddof=1).sum()
    var_total = df_items.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - (var_items / var_total))

# Pengelompokan kolom (Sesuaikan dengan nama kolom di datamu)
dimensi_cols = {
    'X (Variabel Independen)': ['X_1', 'X_2', 'X_3'],
    'Xm (Variabel Moderasi)': ['Xm_1', 'Xm_2', 'Xm_3'],
    'Y (Variabel Dependen)': ['Y_1', 'Y_2', 'Y_3']
}

print("=== 1. UJI VALIDITAS & RELIABILITAS ===")
for nama_dimensi, kolom in dimensi_cols.items():
    print(f"\n--- Menguji {nama_dimensi} ---")

    # 1. Uji Validitas (Korelasi Pearson item ke total)
    skor_total = df[kolom].sum(axis=1)
    for item in kolom:
        r_stat, p_val = pearsonr(df[item], skor_total)
        status = "Valid" if p_val < 0.05 and r_stat > 0.3 else "Tidak Valid"
        print(f"Validitas {item}: r={r_stat:.3f}, p={p_val:.3f} -> {status}")

    # 2. Uji Reliabilitas (Cronbach's Alpha)
    alpha = cronbach_alpha(df[kolom])
    status_alpha = "Reliabel" if alpha > 0.6 else "Tidak Reliabel"
    print(f"Reliabilitas Keseluruhan: Alpha = {alpha:.3f} -> {status_alpha}")

=== 1. UJI VALIDITAS & RELIABILITAS ===

--- Menguji X (Variabel Independen) ---
Validitas X_1: r=0.863, p=0.000 -> Valid
Validitas X_2: r=0.860, p=0.000 -> Valid
Validitas X_3: r=0.852, p=0.000 -> Valid
Reliabilitas Keseluruhan: Alpha = 0.820 -> Reliabel

--- Menguji Xm (Variabel Moderasi) ---
Validitas Xm_1: r=0.825, p=0.000 -> Valid
Validitas Xm_2: r=0.791, p=0.000 -> Valid
Validitas Xm_3: r=0.816, p=0.000 -> Valid
Reliabilitas Keseluruhan: Alpha = 0.738 -> Reliabel

--- Menguji Y (Variabel Dependen) ---
Validitas Y_1: r=0.863, p=0.000 -> Valid
Validitas Y_2: r=0.862, p=0.000 -> Valid
Validitas Y_3: r=0.904, p=0.000 -> Valid
Reliabilitas Keseluruhan: Alpha = 0.849 -> Reliabel


# Agregasi Data dan Mean Centering untuk MRA (Moderated Regression Analysis)

In [4]:
print("\n=== 2. AGREGASI & MEAN CENTERING (WAJIB UNTUK MRA) ===")
df_agg = pd.DataFrame()

# Hitung Rata-rata per dimensi
df_agg['X_avg'] = df[dimensi_cols['X (Variabel Independen)']].mean(axis=1)
df_agg['Xm_avg'] = df[dimensi_cols['Xm (Variabel Moderasi)']].mean(axis=1)
df_agg['Y'] = df[dimensi_cols['Y (Variabel Dependen)']].mean(axis=1)

# Mean Centering: Mengurangi rata-rata individu dengan rata-rata total populasi
df_agg['X_Centered'] = df_agg['X_avg'] - df_agg['X_avg'].mean()
df_agg['Xm_Centered'] = df_agg['Xm_avg'] - df_agg['Xm_avg'].mean()

# Membuat Variabel Interaksi dari data yang sudah di-centering
df_agg['Interaksi'] = df_agg['X_Centered'] * df_agg['Xm_Centered']

print("Data siap diregresikan. Mean Centering selesai.")
df_agg.sample(5)


=== 2. AGREGASI & MEAN CENTERING (WAJIB UNTUK MRA) ===
Data siap diregresikan. Mean Centering selesai.


,X_avg,Xm_avg,Y,X_Centered,Xm_Centered,Interaksi
77,3.666667,2.333333,2.666667,0.206107,-0.857506,-0.176738
100,3.000000,3.333333,4.000000,-0.460560,0.142494,-0.065627
108,4.333333,2.666667,4.666667,0.872774,-0.524173,-0.457484
82,3.666667,3.000000,3.666667,0.206107,-0.190840,-0.039333
63,4.000000,3.666667,3.666667,0.539440,0.475827,0.256680


# Uji Asumsi Klasik

In [5]:
print("\n=== 3. UJI ASUMSI KLASIK (PADA MODEL LENGKAP) ===")
# Siapkan variabel untuk regresi
X_reg = df_agg[['X_Centered', 'Xm_Centered', 'Interaksi']]
Y = df_agg['Y']
X_const = sm.add_constant(X_reg)

# Jalankan model untuk mendapatkan residual
model = sm.OLS(Y, X_const).fit()
residual = model.resid


=== 3. UJI ASUMSI KLASIK (PADA MODEL LENGKAP) ===


In [6]:
# A. Uji Normalitas (Shapiro-Wilk)
stat, p_norm = shapiro(residual)
print(f"1. Normalitas (Shapiro) p-value: {p_norm:.4f} -> {'Aman (Normal)' if p_norm > 0.05 else 'Tidak Normal'}")

# B. Uji Multikolinearitas (VIF)
print("2. Multikolinearitas (VIF):")
for i, col in enumerate(X_const.columns):
    if col != 'const':
        vif = variance_inflation_factor(X_const.values, i)
        # Standar VIF tetap < 10. Berkat Mean Centering, variabel interaksi harusnya aman.
        print(f"   - {col}: {vif:.2f} -> {'Aman' if vif < 10 else 'Terjadi Multikolinearitas'}")

# C. Uji Heteroskedastisitas (Breusch-Pagan)
bp_test = het_breuschpagan(residual, X_const)
print(f"3. Heteroskedastisitas (BP Test) p-value: {bp_test[1]:.4f} -> {'Aman' if bp_test[1] > 0.05 else 'Terjadi Heteroskedastisitas'}")

# D. Uji Linearitas (Rainbow Test)
rainbow_stat, rainbow_p = linear_rainbow(model)
print(f"4. Linearitas (Rainbow Test) p-value: {rainbow_p:.4f} -> {'Aman (Linear)' if rainbow_p > 0.05 else 'Tidak Linear'}")

1. Normalitas (Shapiro) p-value: 0.3984 -> Aman (Normal)
2. Multikolinearitas (VIF):
   - X_Centered: 1.00 -> Aman
   - Xm_Centered: 1.01 -> Aman
   - Interaksi: 1.01 -> Aman
3. Heteroskedastisitas (BP Test) p-value: 0.5589 -> Aman
4. Linearitas (Rainbow Test) p-value: 0.7280 -> Aman (Linear)


# Uji Hipotesis dengan MRA

In [ ]:
# Opsional: Jika kamu ingin langsung melihat hasil hipotesisnya
print("\n=== HASIL UJI HIPOTESIS ===")
print(model.summary())


=== HASIL UJI HIPOTESIS ===
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.629
Method:                 Least Squares   F-statistic:                     74.38
Date:                Mon, 06 Apr 2026   Prob (F-statistic):           7.81e-28
Time:                        04:26:54   Log-Likelihood:                -67.938
No. Observations:                 131   AIC:                             143.9
Df Residuals:                     127   BIC:                             155.4
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           3.717

In [ ]:
# ==========================================
# 4. HASIL HIPOTESIS (MRA)
# ==========================================
print("\n=== TAHAP 3: HASIL REGRESI MODERASI ===")
print(model.summary().tables[1])


=== TAHAP 3: HASIL REGRESI MODERASI ===
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           3.7170      0.036    103.048      0.000       3.646       3.788
X_Centered      0.6302      0.052     12.115      0.000       0.527       0.733
Xm_Centered     0.5660      0.068      8.336      0.000       0.432       0.700
Interaksi       0.1104      0.109      1.012      0.314      -0.106       0.326


# Jawaban

* X_centered memiliki nilai Uji T (P > |t|) 0.000 atau kurang dari 0.05, yang berarti X signifikan berpengaruh terhadap Y. Kita menolak H0 dan menerima H1. X memiliki koefisien positif, yang berarti jika X naik, maka Y juga naik.
* Xm_centered memiliki nilai Uji T (P > |t|) 0.000 atau kurang dari 0.05, yang berarti Xm signifikan berpengaruh terhadap Y. Kita menolak H0 dan menerima H2. Xm memiliki koefisien positif, yang berarti jika X2 naik, maka Y juga naik.
* Hasil Uji F (Prob F-Statistics) 7.81e-28 atau kurang dari 0.05. Pada kasus ini, model RMA masih bersifat baik dan bisa diterima.
* Interaksi memiliki nilai Uji T (P > |t|) 0.314 atau lebih dari 0.05, yang berarti X berpengaruh terhadap Y tidak dimoderasi oleh Xm. Ini berarti kita tetap menerima H0.
* Nilai Adjusted R-Squared adalah 0.629 yang berarti variabel X dan Xm dapat menjelaskan 62.9% variasi yang terjadi pada Y. Sisanya 38.1% dijelaskan oleh faktor lain di luar model ini.


NB: Jika seandainya Interaksi signifikan, maka dilihat dari arah koefisien. Jika positif maka moderasi memperkuat, jika negatif maka moderasi memperlemah